In [1]:
import os
from shared_lib.local import LOCAL_ENV, LOCAL_RUN
from shared_lib.spark import get_spark_session

symbol = "ADAUSDT"
landing_date = "2025-09-27"
data_lake_bucket=os.getenv("DATA_LAKE_BUCKET", "your-data-lake-bucket")
iceberg_lock_table=os.getenv("ICEBERG_LOCK_TABLE", "your_iceberg_lock_table")
transform_db=os.getenv("TRANSFORM_DB", "your_transform_db")
spark = get_spark_session(app_name="dataops-crypto-cloud", master=True, jars=True, local_run=LOCAL_RUN, minio=LOCAL_ENV, hive=LOCAL_ENV, glue=not LOCAL_ENV, iceberg_lock_table=iceberg_lock_table)

26/03/02 11:17:59 WARN Utils: Your hostname, Nguyens-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.19 instead (on interface en0)
26/03/02 11:17:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/anhtu/.ivy2/cache
The jars for the packages stored in: /Users/anhtu/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-677b9705-8e66-4430-bb5e-707c98f8949f;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/anhtu/.pyenv/versions/3.11.11/envs/spark/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.7.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.7.1 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 104ms :: artifacts dl 5ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-aws-bundle;1.7.1 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.7.1 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-------------------------------

In [2]:
from shared_lib.spark import create_database, database_exists

if database_exists(spark, transform_db):
    print("Database exists")
else:
    location=f"s3a://{data_lake_bucket}/transform_zone"
    create_database(spark, transform_db, location=location)
    print("Database created")

Database exists


In [3]:
spark.sql(f"show tables in {transform_db}").show(truncate=False)

+------------+-------------+-----------+
|namespace   |tableName    |isTemporary|
+------------+-------------+-----------+
|transform_db|aggtrades    |false      |
|transform_db|klines       |false      |
|transform_db|pattern_two  |false      |
|transform_db|macd         |false      |
|transform_db|rsi6         |false      |
|transform_db|pattern_one  |false      |
|transform_db|pattern_three|false      |
+------------+-------------+-----------+



In [5]:
spark.sql(f"select * from {transform_db}.pattern_two where pattern is not null order by group_id").show()

+--------+-------------------+----------------+----------+----------+---------+-----------+---------+----------------+------------+-------+------+------+---------+-----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price|   volume|      close_time|landing_date| symbol|  ema7| ema20|    trend|          pattern|
+--------+-------------------+----------------+----------+----------+---------+-----------+---------+----------------+------------+-------+------+------+---------+-----------------+
| 1954413|2025-09-27 11:15:00|1758971700405060|    0.7837|    0.7853|   0.7837|      0.785| 121594.4|1758972591323187|  2025-09-27|ADAUSDT|0.7835| 0.784|downtrend|bullish engulfing|
| 1954445|2025-09-27 19:15:00|1759000502967967|    0.7803|    0.7823|     0.78|     0.7821|  95233.0|1759001384883232|  2025-09-27|ADAUSDT|0.7808|0.7817|downtrend|bullish engulfing|
| 1954493|2025-09-28 07:15:00|1759043701149668|    0.7712|    0.7737|   0.7711|      0.773